# COMFORT Eval — Test02 Analysis

1 external image + 2 POV images as binary A/B options. Correct = POV camera facing toward the object.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

RELATIONS = ["front", "behind", "left", "right"]

RELATION_MAP = {
    "infrontof": "front",
    "totheleft": "left",
    "totheright": "right",
    "behind": "behind",
}

TRUE_LABEL_GROUPS = {
    "front_behind": ["front", "behind"],
    "left_right":   ["left",  "right"],
}

OPPOSITE_RELATION = {"front": "behind", "behind": "front", "left": "right", "right": "left"}

# Test02: correct = always "front" (cam_pov_front).
# When wrong, model picked the distractor whose direction is relation-dependent:
#   front/behind -> cam_pov_back  -> "behind"
#   left         -> cam_pov_left  -> "left"
#   right        -> cam_pov_right -> "right"
TEST02_DISTRACTOR_DIRECTION = {
    "front":  "behind",
    "behind": "behind",
    "left":   "left",
    "right":  "right",
}

# ── Shared helpers ─────────────────────────────────────────────────────────

def extract_camera_perspective_comfort(image_path):
    image_path = str(image_path)
    parts = os.path.dirname(image_path).split("/")
    true_relation = parts[-2]
    folder_name = parts[-1]
    segments = folder_name.split("__")
    cam_segment = segments[-1]
    camera_position = cam_segment.split("_")[-1]

    camera_perspective = None
    if (true_relation == "behind" and camera_position == "left") or (true_relation == "infrontof" and camera_position == "right") or (true_relation == "totheleft" and camera_position == "front") or (true_relation == "totheright" and camera_position == "back"):
        camera_perspective = "right"
    elif (true_relation == "behind" and camera_position == "right") or (true_relation == "infrontof" and camera_position == "left") or (true_relation == "totheleft" and camera_position == "back") or (true_relation == "totheright" and camera_position == "front"):
        camera_perspective = "left"
    elif (true_relation == "behind" and camera_position == "front") or (true_relation == "infrontof" and camera_position == "back") or (true_relation == "totheleft" and camera_position == "right") or (true_relation == "totheright" and camera_position == "left"):
        camera_perspective = "behind"
    elif (true_relation == "behind" and camera_position == "back") or (true_relation == "infrontof" and camera_position == "front") or (true_relation == "totheleft" and camera_position == "left") or (true_relation == "totheright" and camera_position == "right"):
        camera_perspective = "front"
    return true_relation, camera_perspective


def extract_letter_to_relation(prompt):
    prompt = str(prompt)
    mapping = {}
    for letter in ["A", "B", "C", "D"]:
        pattern = rf"{letter}\.\s*(.*?)(?=(?:A|B|C|D)\.\s|$)"
        m = re.search(pattern, prompt, flags=re.IGNORECASE | re.DOTALL)
        if not m:
            continue
        text = m.group(1).lower()
        if "front" in text:    mapping[letter] = "front"
        elif "behind" in text: mapping[letter] = "behind"
        elif "left" in text:   mapping[letter] = "left"
        elif "right" in text:  mapping[letter] = "right"
    return mapping


def plot_confusion_on_ax(df, ax, title, true_labels):
    plot_df = df[
        df["correct_relation"].isin(true_labels) &
        df["pred_relation"].isin(RELATIONS)
    ].copy()

    if len(plot_df) == 0:
        ax.axis("off")
        ax.set_title(f"{title}\n(no data)")
        return

    cm_df = pd.crosstab(
        plot_df["correct_relation"],
        plot_df["pred_relation"],
        normalize="index"
    ).reindex(index=true_labels, columns=RELATIONS, fill_value=0.0)

    im = ax.imshow(cm_df.values, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(RELATIONS)))
    ax.set_xticklabels(RELATIONS)
    ax.set_yticks(range(len(true_labels)))
    ax.set_yticklabels(true_labels)
    for i in range(cm_df.shape[0]):
        for j in range(cm_df.shape[1]):
            ax.text(j, i, f"{cm_df.iloc[i, j]:.2f}", ha="center", va="center")
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")


# ── Test01 loader ─────────────────────────────────────────────────────────

def load_and_prepare_pair(csv_path):
    df = pd.read_csv(csv_path)

    def get_pred_relation(row):
        return extract_letter_to_relation(row["mcq_prompt"]).get(str(row["pred_letter"]).strip(), None)

    df["pred_relation"] = df.apply(get_pred_relation, axis=1)
    df["correct_relation"] = df["correct_relation"].astype(str).str.strip().str.lower()

    r1 = df["image_path_1"].apply(extract_camera_perspective_comfort)
    r2 = df["image_path_2"].apply(extract_camera_perspective_comfort)
    df["object_position"]   = r1.apply(lambda x: RELATION_MAP.get(x[0], x[0]))
    df["cam_perspective_1"] = r1.apply(lambda x: x[1])
    df["cam_perspective_2"] = r2.apply(lambda x: x[1])

    df["cam_view_1"] = df["cam_view_1"].astype(str).str.strip()
    df["cam_view_2"] = df["cam_view_2"].astype(str).str.strip()
    df["view2_short"] = df["cam_view_2"].str.replace("cam_", "", regex=False)

    df["is_correct"]  = df["pred_letter"].astype(str).str.strip() == df["correct_letter"].astype(str).str.strip()
    df["is_opposite"] = df["pred_letter"].astype(str).str.strip() == df["opposite_letter"].astype(str).str.strip()
    return df


# ── Test02 loader ─────────────────────────────────────────────────────────

def add_human_view_scenario(df):
    # In POV tests, cam_pov_front is what the human sees when looking straight ahead.
    # If the target object is in front, the human is looking at the object; otherwise the view is empty/void.
    df["human_view_scenario"] = np.where(
        df["correct_relation"].eq("front"),
        "looking_at_object",
        "looking_at_void",
    )
    return df


def load_and_prepare_pov(csv_path):
    df = pd.read_csv(csv_path)
    df["correct_relation"] = df["correct_relation"].astype(str).str.strip().str.lower()
    df["correct_relation"] = df["correct_relation"].map(RELATION_MAP).fillna(df["correct_relation"])
    df["is_correct"] = df["correct"].astype(str).str.lower().isin(["true", "1"])
    df["cam_view_external_short"] = df["cam_view_external"].str.replace("cam_", "", regex=False)
    return add_human_view_scenario(df)


print("Helpers loaded.")

---
## Test 02 — POV Image-Choice MCQ

Input: 1 external image + 2 POV images as options (binary A/B).  
Correct = POV camera facing toward the object (relation-dependent).  
Distractor = POV camera facing opposite direction.

In [ ]:
TEST02_DIR = "./results"
models02   = ["qwen3vl"]  # ← add more models here as results arrive

### 1 — Confusion matrices by human-view scenario

Split into two sub-scenarios:

- `looking_at_object`: `correct_relation == "front"`; the target object is straight ahead in the human POV.
- `looking_at_void`: `correct_relation != "front"`; looking straight ahead is an empty/void scene without the target object.

Rows = object position (`correct_relation`), columns = model chose correct POV (`cam_pov_front`) vs distractor.


In [ ]:
all_dfs_02 = {}

SCENARIO_RELATIONS_02 = {
    "looking_at_object": ["front"],
    "looking_at_void": ["behind", "left", "right"],
}


def plot_pov_binary_confusion(df, ax, title, relations):
    mat = np.zeros((len(relations), 2))
    annot = []
    for i, rel in enumerate(relations):
        sub = df[df["correct_relation"] == rel]
        n = len(sub)
        if n > 0:
            mat[i, 0] = sub["is_correct"].mean()
            mat[i, 1] = 1 - mat[i, 0]
            annot.append([f"{mat[i,0]:.2f}\n(n={n})", f"{mat[i,1]:.2f}\n(n={n})"])
        else:
            annot.append(["", ""])

    im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["correct\n(cam_pov_front)", "distractor"])
    ax.set_yticks(range(len(relations)))
    ax.set_yticklabels(relations)
    for i in range(len(relations)):
        for j in range(2):
            ax.text(j, i, annot[i][j], ha="center", va="center", fontsize=9)
    ax.set_title(title)
    ax.set_xlabel("Model choice")
    ax.set_ylabel("Object position")
    return im


for model in models02:
    csv_path = os.path.join(TEST02_DIR, f"pov_{model}.csv")
    if not os.path.exists(csv_path):
        print(f"Missing: {csv_path}")
        continue
    df = load_and_prepare_pov(csv_path)
    all_dfs_02[model] = df
    acc = df["is_correct"].mean()
    print(f"Loaded {model}: {len(df)} rows  |  overall acc = {acc:.3f}")
    display(
        df.groupby("human_view_scenario")["is_correct"]
        .agg(n="size", accuracy="mean")
        .assign(accuracy=lambda x: x["accuracy"].round(3))
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, (scenario, relations) in zip(axes, SCENARIO_RELATIONS_02.items()):
        sub = df[df["human_view_scenario"] == scenario]
        im = plot_pov_binary_confusion(
            sub,
            ax,
            title=f"{scenario} (n={len(sub)})",
            relations=relations,
        )
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
    fig.suptitle(f"{model} — Test02: POV binary choice by human-view scenario", fontsize=14)
    plt.tight_layout()
    plt.show()


### 2 — Accuracy heatmap: relation × external camera view

In [ ]:
# EXT_VIEWS = ["back", "front", "left", "right"]

# for model, df in all_dfs_02.items():
#     mat = np.full((len(RELATIONS), len(EXT_VIEWS)), np.nan)
#     annot = []
#     for i, rel in enumerate(RELATIONS):
#         row_ann = []
#         for j, view in enumerate(EXT_VIEWS):
#             sub = df[(df["correct_relation"] == rel) & (df["cam_view_external_short"] == view)]
#             if len(sub) > 0:
#                 mat[i, j] = sub["is_correct"].mean()
#                 row_ann.append(f"{mat[i,j]:.2f}\n(n={len(sub)})")
#             else:
#                 row_ann.append("")
#         annot.append(row_ann)

#     fig, ax = plt.subplots(figsize=(6, 4))
#     sns.heatmap(
#         mat, ax=ax,
#         annot=annot, fmt="",
#         cmap="RdYlGn", vmin=0, vmax=1,
#         linewidths=0.5,
#         xticklabels=EXT_VIEWS,
#         yticklabels=RELATIONS,
#         annot_kws={"size": 9},
#         cbar=True,
#     )
#     ax.set_xlabel("External camera view")
#     ax.set_ylabel("Correct relation")
#     ax.set_title(f"{model} — Test02: accuracy by relation × external camera")
#     plt.tight_layout()
#     plt.show()

### 3 — Per-relation accuracy table (short / middle / long)

In [ ]:
# for model, df in all_dfs_02.items():
#     print(f"\n===== {model} =====")
#     rows = []
#     for rel in RELATIONS:
#         sub = df[df["correct_relation"] == rel]
#         acc = sub["is_correct"].mean() if len(sub) > 0 else np.nan
#         rows.append({"relation": rel, "n": len(sub), "accuracy": round(acc, 3)})
#     rows.append({"relation": "OVERALL", "n": len(df), "accuracy": round(df["is_correct"].mean(), 3)})
#     display(pd.DataFrame(rows).set_index("relation"))